# 💬 实验五：长对话测试

## 学习目标
- 加载微调后的模型
- 通过交互式对话测试多轮回答效果

## 1. 加载微调后的模型（NPU + FlashAttention 加速）

> **硬件说明**：本实验基于 **Ascend NPU** 运行。
> 
> 多轮对话中，随着对话历史增长，输入序列越来越长，SDPA（FlashAttention）的线性显存优势更加明显——标准 Attention 的显存随序列长度**平方增长**，而 SDPA 仅**线性增长**。

In [ ]:
import torch
from transformers import AutoModelForCausalLM, PreTrainedTokenizerFast

MODEL_PATH = "./sft_output"

tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=f"{MODEL_PATH}/tokenizer.json",
    pad_token="</s>",
    eos_token="</s>",
    bos_token="<s>"
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa"  # 🔥 SDPA — 长序列下显存优势更明显
)
model.eval()

print(f"模型已加载: {model.device}")

---

### 🔥 长对话场景下的显存分析

多轮对话时，每次请求都会把完整的对话历史拼接到 prompt 中，序列长度随轮数增长。标准 Attention 的中间矩阵 $S$ 和 $P$ 占用 $O(N^2)$ 显存，而 SDPA（FlashAttention）通过 Tiling 分块将其降至 $O(N)$：

<table style="margin-left: 0; margin-right: auto; border-collapse: collapse; border: 1px solid #ddd;">
  <thead>
    <tr style="background-color: #f2f2f2;">
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">对话轮数</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">序列长度（约）</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">标准 Attention 显存</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">SDPA 显存</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">1 轮</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">128 tokens</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">0.03 MB</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">0.01 MB</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">5 轮</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">512 tokens</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">0.5 MB</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">0.05 MB</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">10 轮</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">1024 tokens</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><strong>2.0 MB</strong></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><strong>0.1 MB</strong></td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">20 轮</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">2048 tokens</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><strong>8.0 MB</strong></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><strong>0.2 MB</strong></td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">40 轮</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">4096 tokens</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><strong>32 MB</strong></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><strong>0.4 MB</strong></td>
    </tr>
  </tbody>
</table>

> 标准 Attention 的显存占用 $O(N^2)$，每轮对话近似翻倍；SDPA 的显存占用 $O(N)$，增长平缓得多。
> 对于 7B 模型，注意力层只是总显存的一部分，但累计节省的显存可以让模型支持更多轮对话或更大的 batch size。

### 本实验的完整优化栈

```
┌──────────────────────────────────────────────────┐
│              DeepSeek-LLM-7B 微调全栈优化           │
├───────────────────┬──────────────────────────────┤
│   LoRA 低秩适配    │  attn_implementation="sdpa"  │
│   (参数优化)       │   (计算优化)                   │
├───────────────────┼──────────────────────────────┤
│   r=8             │   PyTorch SDPA               │
│   参数量 0.06%     │   → FlashAttention 算法       │
│   显存 ~16GB       │   → Tiling + Online Softmax  │
│                   │   → 显存 O(N)、速度 3-8x      │
├───────────────────┴──────────────────────────────┤
│   Ascend NPU 硬件底座                              │
│   torch_npu + 达芬奇架构 Cube Unit                 │
└──────────────────────────────────────────────────┘
```

### 参考文章

- [AIInfraGuide — FlashAttention V1 详解](https://caomaolufei.github.io/AIInfraGuide/guides/模块二-cuda编程与算子优化/61-flashattention-v1详解/)
- [AIInfraGuide — FlashAttention V2 详解](https://caomaolufei.github.io/AIInfraGuide/guides/模块二-cuda编程与算子优化/62-flashattention-v2详解/)

---

## 2. 交互式对话

直接输入问题与模型连续对话，输入 `exit` 退出。

In [ ]:
import torch

def chat(model, tokenizer, history, max_new_tokens=256):
    prompt = ""
    for turn in history:
        prompt += f"<|{turn['role']}|>\n{turn['content']}\n<|end|>\n"
    prompt += "<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    input_len = inputs["input_ids"].shape[1]
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    outputs = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        do_sample=True, temperature=0.7, top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )
    new_tokens = outputs[0][input_len:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=False)
    for stop in ["<|end|>", "<|user|>"]:
        pos = answer.find(stop)
        if pos != -1:
            answer = answer[:pos]
    return answer.strip()

history = []
print("🤖 你好，我是微调后的心理健康助手。输入 'exit' 退出。\n")
while True:
    user_input = input("🙋 你: ").strip()
    if user_input == "exit":
        break
    if not user_input:
        continue
    history.append({"role": "user", "content": user_input})
    answer = chat(model, tokenizer, history)
    history.append({"role": "assistant", "content": answer})
    print(f"🤖 {answer}\n")

## 小结

✅ 成功加载了微调后的 LoRA 模型，支持多轮交互式对话
✅ 全程保持 **SDPA（FlashAttention）算子加速**，长对话场景下显存优势更明显
✅ 理解了 LoRA + FlashAttention 的全栈优化：LoRA 减少参数量，FlashAttention 加速每次计算
✅ 完成了基于 **Ascend NPU** 的完整微调实验流程

---

## 📚 全实验总结

<table style="margin-left: 0; margin-right: auto; border-collapse: collapse; border: 1px solid #ddd;">
  <thead>
    <tr style="background-color: #f2f2f2;">
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">实验</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">内容</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">关键优化</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">实验一</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">环境配置与模型加载</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><strong>FlashAttention(SDPA)</strong> 原理与配置</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">实验二</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">数据探索与预处理</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">多轮对话数据格式化</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">实验三</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">LoRA 微调训练</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><strong>LoRA 低秩分解</strong> + FlashAttention 协同加速</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">实验四</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">模型评测与对话测试</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">推理阶段 SDPA 持续生效</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">实验五</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">长对话测试</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">长序列下 FlashAttention 的显存优势</td>
    </tr>
  </tbody>
</table>

**核心配置**：`attn_implementation="sdpa"` + `LoraConfig(r=8)` → 在 Ascend NPU 上实现高效微调。

## 课后练习

1. (单选题) 构造多轮 prompt 时，历史轮次的正确顺序是？
   - A. system → 第1轮 user/assistant → ... → 最新 user
   - B. 最新 user 放最前
   - C. 随机排列
   - D. 只保留 assistant

2. (单选题) tokenizer 默认 truncation_side="right"，对长对话的含义是？
   - A. 从文本末尾截断，可能丢掉最新 user 输入
   - B. 从开头截断
   - C. 不影响
   - D. 只截断 assistant

3. (多选题) 长对话测试面临？
   - A. 超窗口截断
   - B. 早期上下文遗忘
   - C. KV cache 增大
   - D. 推理时延增加

4. (多选题) 正确构造多轮 history 需要？
   - A. system 只出现一次
   - B. 保留历史 assistant 回答
   - C. 最新 user 提问放最后
   - D. 超长时优先截断最旧轮次

5. (判断题) history 越长，模型回答质量一定越高。

6. (判断题) 将 pad_token_id 设为 eos_token_id 可避免填充导致的生成报错，但可能影响停止行为。

7. (填空题) 控制输入不超过窗口应设置 max_length 与 ____。

8. (填空题) KV cache 大小随序列长度近似 ____ 增长。

9. (简答题) 为什么超长对话优先保留近期上下文，而不是均等保留所有轮次？

10. (简答题) 如何验证模型记住了第 1 轮的细节？

11. (代码设计题) 编写 chat_once(history, user_input)，构造完整 prompt，并返回 (prompt, 更新后的 history)。

12. (单选题) 生成结果中出现 <|user|>，正确处理是？
   - A. 在该标记处截断回答
   - B. 保留
   - C. 报错
   - D. 忽略

13. (多选题) 长对话评测维度包括？
   - A. 上下文一致性
   - B. 追问能力
   - C. 安全边界
   - D. 建议相关性

14. (判断题) 多轮评测应固定 seed、生成参数与提示模板，才能公平对比。

15. (简答题) 估算 10 轮对话 token 开销并给出降低开销的方案。

> 参考答案见 answer/05.06_long_dialogue_test_answer.ipynb。